# GAE local exploration

Pre-wired to the same `src/` code your Glue job uses. Runs in the Glue 5.0
container (`docker compose up gae-lab`, then open http://localhost:8888) or
inside the Dev Container.

Start on the **local fixtures**; flip the flags near the bottom to hit the real
Glue Catalog + S3 once the fixture loop looks right.

In [ ]:
# Make `src` importable when the notebook lives in ./notebooks
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.insert(0, os.path.abspath(os.getcwd()))

In [ ]:
# Start Spark with the GAE plugin and authenticate the license (reads .env).
from src import bootstrap_spark, transform, io_glue

spark = bootstrap_spark.start("explore")
spark

In [ ]:
# Sanity check the GAE plugin: ST_Distance((0,0),(3,4)) should be 5.0
spark.sql("SELECT ST_Distance(ST_Point(0,0), ST_Point(3,4)) AS d").show()

## Load local fixtures

In [ ]:
from pyspark.sql.functions import expr

FIX = os.path.abspath(os.path.join(os.getcwd(), "..", "tests", "fixtures"))

rc = spark.read.csv(os.path.join(FIX, "roadcalls_sample.csv"), header=True)
points = transform.add_point_geometry(rc, "longitude", "latitude", 4326)

tracts = spark.read.csv(os.path.join(FIX, "census_tract_sample.csv"), header=True)
tracts = tracts.withColumn("geometry", expr("ST_GeomFromText(wkt, 4326)"))

points.show(truncate=False)

## Run the transform

In [ ]:
TRACT_COLS = [
    "STATE_ABBR", "STATE_FIPS", "COUNTY_FIP", "STCOFIPS",
    "TRACT_FIPS", "FIPS", "POPULATION", "POP_SQMI", "POPULATI_1", "POP20_SQMI",
]

result = transform.roadcalls_per_tract(points, tracts, TRACT_COLS)
result.drop("geometry").show(truncate=False)

## Switch to the real backend

Restart the kernel, then rebuild Spark with the catalog enabled and read from
S3 / Glue Catalog instead of fixtures. Requires AWS creds mounted (they are, via
compose) and Glue/Lake Formation read permissions.

```python
spark = bootstrap_spark.start("explore", enable_glue_catalog=True)

rc = io_glue.read_roadcalls_from_csv(spark, os.environ["ROADCALL_CSV_S3"])
points = transform.add_point_geometry(rc, "longitude", "latitude", 4326)
tracts = io_glue.read_census_tracts_shapefile(spark, os.environ["CENSUS_TRACT_S3"])

result = transform.roadcalls_per_tract(points, tracts, TRACT_COLS)
result.drop("geometry").show()
```